In [1]:
from proptrainer import features
import csv
import json
import pandas as pd
import numpy as np

from google.cloud import storage
from google.cloud import bigquery
from google.cloud import bigquery_storage

segment = 106

In [2]:
inp_features = features.feature_lookup[str(segment)]
inp_features = ','.join([item for item in inp_features])

In [3]:
query = f"""SELECT {inp_features}
FROM `dw-bq-data-d00.SANDBOX_ANALYTICS.dm_pc_refresh_eval_data_w_margin`
WHERE em_segment = {segment}
AND IN_HOME_DT > DATE'2021-03-31'
ORDER BY COUPON_BARCODE
LIMIT 50
"""

bq_client = bigquery.Client(project='dw-bq-data-d00')
bqstorageclient = bigquery_storage.BigQueryReadClient()

sample = bq_client.query(query).result().to_dataframe(
    bqstorage_client=bqstorageclient)

sample.head()

,BBB_F_DECILE_2Y,BBB_ONCOUPON_R_DECILE_2Y,COUPON_Q_01,COUPON_Q_04,NUM_PERIODS,NUM_QUARTERS,NUM_TOTAL_ITEMS,PH_MREDEEM182D_PERC,PH_MREDEEM730D_PERC,PH_PREDEEM365D,...,TOTAL_SALES_L12M,BUYS_Q_01,BUYS_Q_02,BUYS_M_01,BUYS_M_06,BUYS_Q_04,BUYS_Q_08,NUM_TXNS,is_in_trade_area,A_A9350N_ECONOMIC_STB_01_10
0,2.000000000,5.000000000,1,0,1,1,3,0.166667,0.160000,1,...,79.960000000,1,1,1,1,0,2,1,0,0E-9
1,2.000000000,2.000000000,0,0,4,3,14,0.000000,0.000000,0,...,1111.330000000,0,0,0,0,0,1,5,0,0E-9
2,2.000000000,5.000000000,0,2,3,2,18,0.066667,0.023810,0,...,288.870000000,0,1,0,1,2,1,5,0,0E-9
3,1.000000000,5.000000000,0,0,2,1,2,0.000000,0.066667,0,...,18.980000000,0,0,0,0,0,2,2,1,1.000000000
4,2.000000000,2.000000000,2,0,3,2,6,0.090909,0.081633,0,...,198.140000000,2,1,0,1,0,1,4,1,1.000000000


In [4]:
columns = [col for col in sample.columns]
for column in columns:
    if column in [
            'BBB_F_DECILE_2Y', 'BBB_ONCOUPON_R_DECILE_2Y',
            'A_A9350N_ECONOMIC_STB_01_10', 'BBB_INSTORE_F', 'BBB_INSTORE_F_2Y',
            'TOTAL_TXNS_L12M', 'BBB_R_2Y'
    ]:
        sample[column] = sample[column].astype('int')
    if column in [
            'COUPON_ANY_AMT', 'COUPON_SALES_Q_05', 'COUPON_SALES_Q_08',
            'HARMON_SALES_L12M', 'TOTAL_SALES_L12M', 'Z_AVG_PCT_SALES_RETURNED'
    ]:
        sample[column] = sample[column].astype('float')
    if column in [
            'COUPON_SALES_Q_05', 'COUPON_SALES_Q_08', 'TOTAL_SALES_L12M',
            'NUM_TOTAL_ITEMS'
    ]:
        sample.loc[(sample[column] < 0), column] = 0

In [5]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   BBB_F_DECILE_2Y              50 non-null     int64  
 1   BBB_ONCOUPON_R_DECILE_2Y     50 non-null     int64  
 2   COUPON_Q_01                  50 non-null     int64  
 3   COUPON_Q_04                  50 non-null     int64  
 4   NUM_PERIODS                  50 non-null     int64  
 5   NUM_QUARTERS                 50 non-null     int64  
 6   NUM_TOTAL_ITEMS              50 non-null     int64  
 7   PH_MREDEEM182D_PERC          50 non-null     float64
 8   PH_MREDEEM730D_PERC          50 non-null     float64
 9   PH_PREDEEM365D               50 non-null     int64  
 10  PH_PREDEEM548D               50 non-null     int64  
 11  PH_PREDEEM730D               50 non-null     int64  
 12  BBB_R_2Y                     50 non-null     int64  
 13  A_A8642_HM_MKT_VAL    

In [6]:
sample.head()

,BBB_F_DECILE_2Y,BBB_ONCOUPON_R_DECILE_2Y,COUPON_Q_01,COUPON_Q_04,NUM_PERIODS,NUM_QUARTERS,NUM_TOTAL_ITEMS,PH_MREDEEM182D_PERC,PH_MREDEEM730D_PERC,PH_PREDEEM365D,...,TOTAL_SALES_L12M,BUYS_Q_01,BUYS_Q_02,BUYS_M_01,BUYS_M_06,BUYS_Q_04,BUYS_Q_08,NUM_TXNS,is_in_trade_area,A_A9350N_ECONOMIC_STB_01_10
0,2,5,1,0,1,1,3,0.166667,0.160000,1,...,79.96,1,1,1,1,0,2,1,0,0
1,2,2,0,0,4,3,14,0.000000,0.000000,0,...,1111.33,0,0,0,0,0,1,5,0,0
2,2,5,0,2,3,2,18,0.066667,0.023810,0,...,288.87,0,1,0,1,2,1,5,0,0
3,1,5,0,0,2,1,2,0.000000,0.066667,0,...,18.98,0,0,0,0,0,2,2,1,1
4,2,2,2,0,3,2,6,0.090909,0.081633,0,...,198.14,2,1,0,1,0,1,4,1,1


In [7]:
storage_client = storage.Client(project='dw-analytics-d01')

filename = "sample.csv"
sample.to_csv(filename, index=False, encoding='utf-8')

gcs_path = f"gs://ai-ml-vertex-d01/dm-propensity/pc/{segment}/data/{filename}"
blob = storage.blob.Blob.from_string(gcs_path, client=storage_client)
blob.upload_from_filename(filename)